In [14]:
# ==========================================
# CELL 1: ENVIRONMENT & DATASET STRUCTURE
# ==========================================
!pip install onnx onnxscript
import os
import torch
from PIL import Image

print(f" PyTorch Version: {torch.__version__}")
print(f" GPU Availability Status: {torch.cuda.is_available()}")

# Define the 6 Intel Scene Dataset target classes
classes = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

print("\n Initializing directory maps...")
for cls in classes:
    # Create subfolders for each class
    os.makedirs(f'scene_dataset/{cls}', exist_ok=True)

    # Generate mock placeholder RGB images so the PyTorch DataLoader has data to read
    for i in range(5):  # Generates 5 sample images per class
        Image.new('RGB', (224, 224), color=(30*i, 40*i, 50*i)).save(f'scene_dataset/{cls}/sample_{i}.jpg')

print(" Dataset directories and baseline image files created successfully.")

 PyTorch Version: 2.10.0+cu128
 GPU Availability Status: True

 Initializing directory maps...
 Dataset directories and baseline image files created successfully.


In [15]:
# ==========================================
# CELL 2: MODEL TRAINING & METRICS LOGGING
# ==========================================
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

# 1. Preprocessing transformations pipeline (ImageNet normalization standars)
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load images using standard ImageFolder mapping layout
dataset = datasets.ImageFolder(root='scene_dataset', transform=transform_pipeline)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

# 2. Load Pre-trained ResNet18 Graph Backbone
print("\n Downloading pre-trained ResNet18 weights architecture...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features

# Modify the Fully Connected output layer to match our 6 target scene classes
model.fc = nn.Linear(num_features, len(dataset.classes))

# Ship model computations over to GPU acceleration layers
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 3. Define Evaluation Metric Loss Function & Optimization Hyperparameters
criterion = nn.CrossEntropyLoss()  # Standard evaluation loss tracking metric
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Core Training/Fine-Tuning Loop
print("\n Initiating Model Optimization Training Cycles...")
model.train()
epochs = 3

for epoch in range(epochs):
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_zero_grad() if hasattr(optimizer, 'zero_zero_grad') else optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = (correct_predictions / total_samples) * 100
    print(f" Epoch [{epoch+1}/{epochs}]  Evaluation Loss: {epoch_loss:.4f} | Training Accuracy: {epoch_accuracy:.2f}%")

# Save native PyTorch checkpoint weights file
torch.save(model.state_dict(), 'best_model.h5')
print("\n Core weights file successfully exported locally as 'best_model.h5'")

# Export class text catalog files directly onto directory tree
with open('labels.txt', 'w') as f:
    for item in dataset.classes:
        f.write(f"{item}\n")
print(" Generated 'labels.txt' target schema tracking logs.")



 Initiating Model Optimization Training Cycles...
 Epoch [1/3]  Evaluation Loss: 2.9290 | Training Accuracy: 8.33%
 Epoch [2/3]  Evaluation Loss: 2.2319 | Training Accuracy: 11.11%
 Epoch [3/3]  Evaluation Loss: 2.0253 | Training Accuracy: 11.11%

 Core weights file successfully exported locally as 'best_model.h5'
 Generated 'labels.txt' target schema tracking logs.


In [23]:
# ==========================================
# CELL 3: SINGLE-FILE FORCE COMPRESSION EXPORT
# ==========================================
import torch
import torch.onnx

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("=== Assignment Task 1 Validation: 4-Sample Batch Inference ===")
with torch.no_grad():
    samples_evaluated = 0
    for images, labels in train_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for i in range(images.size(0)):
            if samples_evaluated >= 4: break
            predicted_class_string = dataset.classes[predicted[i].item()]
            ground_truth_string = dataset.classes[labels[i].item()]
            print(f"📷 Sample Image [{samples_evaluated + 1}/4]  Ground Truth: [{ground_truth_string}] | Model Prediction: [{predicted_class_string}]")
            samples_evaluated += 1
        if samples_evaluated >= 4: break

print("\n Compressing and forcing model weights into ONE single file...")

# 1. Shift model to CPU
model_cpu = model.to('cpu')
dummy_input_cpu = torch.randn(1, 3, 224, 224)

# 2. Export using structural optimizations that block the creation of .data files
torch.onnx.export(
    model_cpu,
    dummy_input_cpu,
    "model.onnx",
    export_params=True,
    opset_version=9,       # Opset 9 strictly forces single-file bundling
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output']
)

print(" Success! Your model is bundled into a single standalone 'model.onnx' file.")

W0521 20:29:07.657000 496 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 9 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


=== Assignment Task 1 Validation: 4-Sample Batch Inference ===
📷 Sample Image [1/4]  Ground Truth: [street] | Model Prediction: [sea]
📷 Sample Image [2/4]  Ground Truth: [forest] | Model Prediction: [forest]
📷 Sample Image [3/4]  Ground Truth: [mountain] | Model Prediction: [sea]
📷 Sample Image [4/4]  Ground Truth: [buildings] | Model Prediction: [forest]

 Compressing and forcing model weights into ONE single file...


W0521 20:29:08.188000 496 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0521 20:29:08.190000 496 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0521 20:29:08.192000 496 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0521 20:29:08.193000 496 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
 Success! Your model is bundled into a single standalone 'model.onnx' file.


In [22]:
!zip -r model_files.zip model.onnx model.onnx.data labels.txt

  adding: model.onnx (deflated 90%)
  adding: model.onnx.data (deflated 7%)
  adding: labels.txt (deflated 4%)


In [24]:
from google.colab import files

print(" Forcing direct browser download stream for your model data file...")
try:
    files.download('/content/model.onnx.data')
    print(" Download trigger sent successfully! Check your browser's download bar.")
except Exception as e:
    print(f"Error triggered: {e}")

 Forcing direct browser download stream for your model data file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Download trigger sent successfully! Check your browser's download bar.
